In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [3]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_classification.dat'
# 예측 결과를 저장할 파일 이름
prediction_path = 'model/classification_prediction.csv'

### 저장한 모델 객체 등을 복원한다.

In [5]:
# 모델 불러오기
with open('final_model.pkl', 'rb') as f:
    model1 = pickle.load(f)

# 레이블 인코더 불러오기
with open('label_encoder.pkl', 'rb') as f:
    encoder1 = pickle.load(f)

# 스케일러 불러오기 (있다면)
with open('scaler.pkl', 'rb') as f:
    scaler1 = pickle.load(f)

# 확인
display(model1)
display(encoder1)
display(scaler1)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, gpu_id=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.3, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, n_estimators=300, n_jobs=None,
              num_class=5, num_parallel_tree=None, objective='multi:softmax', ...)

LabelEncoder()

StandardScaler()

### 예측할 데이터를 준비한다.
- 모든 준비가 완료되면 입력데이터는 test_X 라는 변수에 담아주도록 한다.

In [51]:
# 예측할 데이터를 읽어온다.
df1 = pd.read_csv("C:/Users/이민정/Desktop/부트캠프/파이널프로젝트/전처리/상위20개컬럼Test.csv")
df1 

,ID,이용금액_R3M_신용체크,_2순위카드이용금액,_1순위업종_이용금액,정상입금원금_B5M,이용금액_오프라인_B0M,_2순위업종_이용금액,최대이용금액_일시불_R12M,연체입금원금_B0M,_3순위쇼핑업종_이용금액,...,정상입금원금_B2M,_3순위업종_이용금액,_1순위교통업종_이용금액,이용건수_신용_R12M,쇼핑_도소매_이용금액,이용금액_오프라인_R6M,청구금액_B0,청구금액_R6M,평잔_일시불_3M,잔액_일시불_B0M
0,TEST_00000,18938,4637,4621,3680,9741,1999,22792,727,648,...,3768,1680,1402,110,0,59101,4764,26492,7074,3505
1,TEST_00001,11757,0,9239,8726,3490,1094,1564,0,0,...,10652,1061,1061,54,0,23830,27643,89983,1056,1145
2,TEST_00002,40339,10227,11110,11297,10759,5578,20151,0,1297,...,10796,2756,2236,525,1653,65643,12136,74074,6739,2516
3,TEST_00003,7559,0,1997,1375,965,1494,1641,0,370,...,1266,705,1271,317,636,5799,2368,12439,2129,959
4,TEST_00004,12863,7696,1237,3951,985,1162,2529,0,460,...,1068,917,683,182,810,11965,1031,12830,1205,483
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,TEST_99995,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99996,TEST_99996,3110,0,972,302,407,0,1067,0,0,...,252,0,0,7,0,2331,359,2237,237,191
99997,TEST_99997,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99998,TEST_99998,173263,31916,64203,6215,15388,16296,64459,16746,1965,...,2751,4612,2584,933,2208,89973,21273,108420,17039,17916


In [53]:
# ID 컬럼을 따로 보존하고, 입력용 데이터에서는 제거한다
ids = df1['ID']
df1 = df1.drop(columns=['ID'])

In [55]:
# 입력 데이터에 대한 표준화
scaled_data = scaler1.transform(df1)
scaled_data

array([[ 0.08543701,  0.16353976, -0.08508045, ..., -0.13332786,
         0.39985056,  0.0314596 ],
       [-0.22328957, -0.43770773,  0.43616494, ...,  1.16443616,
        -0.41361046, -0.33613144],
       [ 1.00551186,  0.88835611,  0.64734947, ...,  0.8392542 ,
         0.35456817, -0.12258597],
       ...,
       [-0.7287482 , -0.43770773, -0.60666447, ..., -0.67482767,
        -0.55635138, -0.5144754 ],
       [ 6.72019951,  3.70061763,  6.64009132, ...,  1.54129076,
         1.74683281,  2.2761013 ],
       [-0.5990839 ,  0.17974764, -0.60384266, ..., -0.67268146,
        -0.55513484, -0.507622  ]])

In [57]:
# 입력데이터를 test_X 변수에 담아준다.
test_X = scaled_data

### 예측하고 저장한다.

In [60]:
# 예측한다.
y_pred = model1.predict(test_X)
y_pred

array([4, 4, 4, ..., 4, 4, 4])

In [61]:
# 결과데이터를 복원한다.
result = encoder1.inverse_transform(y_pred)
result

array(['E', 'E', 'E', ..., 'E', 'E', 'E'], dtype=object)

In [64]:
# 결과를 붙힌다음 저장한다.
df1['Segment'] = result
df1['ID'] = ids
df1[['ID', 'Segment']].to_csv(prediction_path, index=False, encoding='utf-8-sig')
print('저장완료')

저장완료


In [66]:
df_load = pd.read_csv("model/classification_prediction.csv")
df_load

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,E
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_99995,E
99996,TEST_99996,E
99997,TEST_99997,E
99998,TEST_99998,E
